# Re-analysis of a IX83 unit 2 experiment

Re-process an experiment that is already on disk, **offline**, without
the microscope, for example an ERK-KTR run from
`04_erk_experiments_full_fov_stim`. Use it to:

- **Re-segment** with another segmentator (`USE_OLD_SEGMENTATIONS = False`).
- **Re-track** with other tracker settings.
- **Re-extract features**, e.g. `FE_ErkKtr` instead of `SimpleFE`, or add a reference-image extractor.
- **Re-compute stimulation masks** with another stimulator (`USE_OLD_STIM_MASKS = False`).
- **Reuse the old labels and masks** and only re-run tracking and features, which is much faster than re-segmenting.

Raw images are read from the original folder and never changed. New
labels, tracks and features go to a new output folder.

## Raw images

Raw images are **not rewritten**. The output folder gets **hard links**
to the raw data of the source:
- TIFF source: every `raw/*.tiff` file is hard-linked into `out_path/raw/`.
- OME-Zarr source: the resolution levels (`0/`, `1/`, ...) are hard-linked into the new store; labels are written fresh.

A hard link is a second name for the same data on disk: no bytes are
copied, and deleting one name leaves the other. On file systems
without hard links (e.g. SMB shares) the pipeline copies instead.

## Key parameters

| Parameter | Effect |
|-----------|--------|
| `img_storage_path` | the original experiment, with `events.json` and `raw/` or `acquisition.ome.zarr` |
| `out_path` | output folder of the re-analysis |
| `events` | the acquisition plan, restored from `events.json` |
| `use_old_segmentations` | `True`: read the existing label masks instead of segmenting |
| `use_old_stim_masks` | `True`: copy the existing stim masks instead of re-computing them |
| `n_jobs` | parallel threads, one per FOV |
| `correct_timestep_jumps` | `True`: fill missing timesteps so tracking stays continuous |

## 1. Imports and paths

In [ ]:
import os

import pandas as pd

from faro.core.conversion import load_events_json
from faro.core.data_structures import SegmentationMethod
from faro.core.writers import OmeZarrWriter, TiffWriter

In [ ]:
SRC_PATH = None  # TODO: e.g. r"..\04_erk_experiments_full_fov_stim\data\test"
assert SRC_PATH is not None and os.path.isdir(SRC_PATH), "set SRC_PATH"
OUT_PATH = SRC_PATH + "_re"

events = load_events_json(SRC_PATH)

# The output uses the format of the source: an OME-Zarr source gives a
# store with hard-linked raw data, a TIFF source TIFF folders.
if os.path.isdir(os.path.join(SRC_PATH, "acquisition.ome.zarr")):
    writer = OmeZarrWriter(OUT_PATH)
else:
    writer = TiffWriter(OUT_PATH)
print(f"{len(events)} events, writer: {type(writer).__name__}")

## 2. Configure the pipeline

Every component can be swapped on its own:

- **Segmentator**: with `USE_OLD_SEGMENTATIONS = True` the existing masks are read and `DummySegmentator` is never called. Set it to `False` and use e.g. `CellposeV4()` to re-segment.
- **Feature extractor**: `FE_ErkKtr` for the ERK-KTR ratio (`cnr`), `SimpleFE` for area and intensity.
- **Tracker**: `TrackerTrackpy` links the cells across frames.
- **Reference extractor**: `RefFE` measures the optocheck channel on the reference frames.
- **Stimulator**: only needed with `USE_OLD_STIM_MASKS = False`, e.g. `StimWholeFOV()`.

In [ ]:
from faro.core.pipeline_post import ImageProcessingPipeline_postExperiment
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.ref import RefFE
from faro.segmentation.base import DummySegmentator
from faro.tracking.trackpy import TrackerTrackpy

USE_OLD_SEGMENTATIONS = True
USE_OLD_STIM_MASKS = True

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=DummySegmentator(),
        use_channel=0,
        save_tracked=True,
    ),
]

pipeline = ImageProcessingPipeline_postExperiment(
    img_storage_path=SRC_PATH,
    out_path=OUT_PATH,
    events=events,
    segmentators=segmentators,
    feature_extractor=FE_ErkKtr("labels"),
    tracker=TrackerTrackpy(search_range=50),
    feature_extractor_ref=RefFE(used_mask="labels"),
    stimulator=None,
    use_old_segmentations=USE_OLD_SEGMENTATIONS,
    use_old_stim_masks=USE_OLD_STIM_MASKS,
    n_jobs=4,
    writer=writer,
)

## 3. Run the re-analysis

`pipeline.run()` processes the FOVs in parallel. For every frame it:

1. reads the raw image (from `raw/` or `acquisition.ome.zarr`),
2. segments it, or loads the existing masks,
3. tracks the cells,
4. extracts the features and merges them into the tracks,
5. re-computes the stimulation masks if asked,
6. writes everything through the writer,
7. and saves `events.json` to the output folder.

`pipeline.concat_fovs()` merges the per-FOV parquets into one
`exp_data.parquet`.

```
out_path/
├── acquisition.ome.zarr/   (OME-Zarr)  or  raw/, labels/, ...  (TIFF)
├── tracks/                  per-FOV parquet files
├── exp_data/                exp_data.parquet
├── exp_data.parquet         (legacy location)
└── events.json              acquisition plan
```

In [ ]:
pipeline.run()
pipeline.concat_fovs()

## 4. Results

In [ ]:
exp_data = pd.read_parquet(os.path.join(OUT_PATH, "exp_data.parquet"))
n_tracks = exp_data.groupby(["fov", "particle"]).ngroups
print(f"{n_tracks} tracks, {len(exp_data)} rows")
if "cnr" in exp_data:
    ax = exp_data.groupby("timestep")["cnr"].mean().plot(marker="o")
    ax.set_xlabel("timestep")
    ax.set_ylabel("mean ERK-KTR C/N ratio")
exp_data.head()